# Characterization of the Cascade NAS deployed on the XEM7310 Artix-7 or XEM6310 Spartan 6 board using the OKAERTool

This notebook characterizes the frequency response of the Neuromorphic Auditory Sensor (NAS) deployed on the FPGA platform. The OKAERTool (deployed on the same FPGA board) is used both to **configure** the NAS and to **capture** the AER events it produces, through the `pyOKAERTool` Python package. The OKAERTool IP is an open-source hardware platform to deploy and test AER-based systems on the edge.

The notebook is organized in two parts:

1. **Platform set-up and NAS configuration**: initialize the OKAERTool, load the filter bank exported from pyOpenNAS and write it into the NAS registers.
2. **Frequency-sweep characterization**: play a series of pure tones of increasing frequency through the sound card, record the spikes generated by every NAS channel and compute the resulting frequency response (tuning curves, centre frequencies and tracking error). The raw results are stored in a `.mat` file and analysed afterwards.

## Part 1: Platform set-up and NAS configuration
### Initialize the OKAERTool
The cell below loads the FPGA bitfile (which contains the stereo, 64-channel Cascade NAS together with the OKAERTool), initializes the OKAERTool and creates the `pyNAVIS` settings used to interpret the AER events received from the NAS. Select the bitfile that matches your board (XEM7310 Artix-7 or XEM6310 Spartan-6), or set `BITFILE_PATH = None` to skip programming the FPGA.

In [ ]:
import os
import re
import sys
import time

import numpy as np

# Add the parent directory to the path to import pyOKAERTool
sys.path.insert(0, os.path.abspath('..'))
import pyOKAERTool as okt
from pyNAVIS import *

# Bitfile to program (None -> do not program the FPGA)
BITFILE_PATH = '../../bitfiles/CNAS_okaertool_XEM6310.bit'
# BITFILE_PATH = '../../bitfiles/CPNAS_okaertool_XEM7310.bit'
# BITFILE_PATH = None

if BITFILE_PATH is not None and not os.path.exists(BITFILE_PATH):
    raise FileNotFoundError(f"Bitfile not found: {BITFILE_PATH}")

# Create and initialize the OKAERTool
okaer = okt.Okaertool(bit_file=BITFILE_PATH)
okaer.init()

# pyNAVIS settings (stereo, 64 channels, ON/OFF polarity, 10 ns timestamp tick)
settings = MainSettings(num_channels=64, mono_stereo=1, on_off_both=1, address_size=4, ts_tick=0.01, bin_size=10000)

### Configure the Cascade NAS
The NAS must be configured before the characterization. The configuration is written with the `set_config` method of the OKAERTool class, one 16-bit value per register, and is made of:
- `PDM2Spikes_DEFAULT_parameter`: PDM-to-spikes front-end parameters (left and right cochlea, registers `0x00`-`0x07`).
- `I2S2Spikes_DEFAULT_parameter`: I2S-to-spikes front-end parameters (register `0x08`).
- `CASCADE_FILTER_DEFAULT_parameter`: four values per filter (`FREQ_DIV`, `SPIKES_DIV_FB`, `SPIKES_DIV_OUT`, `SPIKES_DIV_BPF`) for each of the 64 channels, written for the left (from `0x09`) and right (from `0x10D`) cochleae.

The filter parameters are parsed from the VHDL config file (`CFBank_*.vhd`) exported by the pyOpenNAS tool, which only contains the Cascade NAS parameters. After resetting the board, the next cell parses that file and writes all the parameters to the NAS (`port_a`). Only the Cascade NAS is characterized here.

In [ ]:
CONFIG_FILE_PATH = './CFBank_64_20_22000.vhd'

PDM2Spikes_DEFAULT_parameter = [0x0005, 0x0006, 0x734B, 0x39C8]
I2S2Spikes_DEFAULT_parameter = [0x000F]

# Register map of the NAS configuration space
PDM2SPIKES_ADDR = 0x0000   # left cochlea, followed by right cochlea
I2S2SPIKES_ADDR = 0x0008
FILTERS_LEFT_ADDR = 0x0009
FILTERS_RIGHT_ADDR = 0x010D


def _tok_to_int(tok):
    """Convert a VHDL literal (x"..", 0x.., 16#..# or decimal) to int."""
    tok = tok.strip().rstrip(',').strip()
    if tok.lower().startswith('x"') and tok.endswith('"'):
        return int(tok[2:-1], 16)
    if tok.lower().startswith('0x'):
        return int(tok, 16)
    m = re.match(r'16#([0-9A-Fa-f]+)#', tok)
    if m:
        return int(m.group(1), 16)
    if tok.isdigit():
        return int(tok, 10)
    raise ValueError(f"Unrecognized token: {tok!r}")


def parse_cascade_vhd(path):
    """Return the flat list [FREQ_DIV, SPIKES_DIV_FB, SPIKES_DIV_OUT, SPIKES_DIV_BPF] * n_filters."""
    with open(path, 'r', encoding='utf-8', errors='ignore') as f:
        text = f.read()
    pattern = re.compile(
        r'FREQ_DIV\s*=>\s*(?P<f>[^,\n;]+)\s*,\s*'
        r'SPIKES_DIV_FB\s*=>\s*(?P<fb>[^,\n;]+)\s*,\s*'
        r'SPIKES_DIV_OUT\s*=>\s*(?P<out>[^,\n;]+)\s*,\s*'
        r'SPIKES_DIV_BPF\s*=>\s*(?P<bpf>[^,\n;]+)',
        re.IGNORECASE | re.DOTALL)
    values = []
    for m in pattern.finditer(text):
        values.extend(_tok_to_int(m.group(g)) for g in ('f', 'fb', 'out', 'bpf'))
    return values


def write_registers(port, start_address, values):
    """Write consecutive registers starting at start_address and return the next free address."""
    for offset, value in enumerate(values):
        okaer.set_config(port, start_address + offset, value)
    return start_address + len(values)


if not os.path.exists(CONFIG_FILE_PATH):
    raise FileNotFoundError(CONFIG_FILE_PATH)
CASCADE_FILTER_DEFAULT_parameter = parse_cascade_vhd(CONFIG_FILE_PATH)
n_filters = len(CASCADE_FILTER_DEFAULT_parameter) // 4
print(f"Parsed {n_filters} filters ({len(CASCADE_FILTER_DEFAULT_parameter)} values) from {CONFIG_FILE_PATH}")

# Reset the OKAERTool (internal logic and NAS) and configure the NAS
okaer.reset_board(mode='both')

okaer.logger.info("Configuring PDM2Spikes modules (left and right cochlea)")
next_addr = write_registers('port_a', PDM2SPIKES_ADDR, PDM2Spikes_DEFAULT_parameter)   # left
write_registers('port_a', next_addr, PDM2Spikes_DEFAULT_parameter)                     # right

okaer.logger.info("Configuring I2S2Spikes modules")
write_registers('port_a', I2S2SPIKES_ADDR, I2S2Spikes_DEFAULT_parameter)

okaer.logger.info("Configuring Cascade NAS filters (left cochlea)")
write_registers('port_a', FILTERS_LEFT_ADDR, CASCADE_FILTER_DEFAULT_parameter)
okaer.logger.info("Configuring Cascade NAS filters (right cochlea)")
write_registers('port_a', FILTERS_RIGHT_ADDR, CASCADE_FILTER_DEFAULT_parameter)

## Part 2: Frequency-sweep characterization of the NAS
### Experiment description
The goal is to measure the **frequency response of every channel of the Cascade NAS**, i.e. how many spikes per second each channel emits as a function of the frequency of the audio stimulus. The experiment works as follows:

1. **Stimulus**: `NUM_FREQS` (128) pure tones are generated, logarithmically spaced between `START_FREQ` (16 Hz) and `END_FREQ` (28 kHz) so that they cover the whole audible range with the same resolution per octave. Each tone is a stereo sine wave (identical on both channels) with amplitude `GAIN`, sampled at `FS` = 96 kHz. The stimulus is played by the host sound card (`sounddevice`) and fed to the NAS audio input.
2. **Acquisition**: for each tone the board is reset (`reset_board(mode='internal')`) so that every acquisition starts from the same state with the NAS timestamps restarted, the tone starts playing, and after `SETTLE_TIME` seconds (to skip the playback start-up transient) the OKAERTool records the NAS output for `MONITOR_TIME` seconds.
3. **Per-channel firing rate**: the AER address encodes the channel (`address // 2`; the lowest address bit is the spike polarity) and the first `NAS_CHANNELS` channels belong to the left cochlea and the rest to the right one. The firing rate of each channel is its spike count divided by the time spanned by the recording (timestamps use a 100 MHz clock, i.e. 10 ns per tick). Channels that did not fire are set to a small floor value (`RATE_FLOOR`) so the results can be shown on logarithmic scales.
4. **Dropout detection and retry**: occasionally the USB transfer or the sound card loses data, producing recordings with an abnormally low event rate ("calvas" or gaps). A recording whose total event rate is below `DROP_RATIO` (80 %) of the previous tone is discarded and repeated up to `MAX_RETRIES` times; if the rate is still low after the last attempt the data is stored anyway.
5. **Output**: the matrices `tfL` and `tfR` (one row per tone, one column per channel), the list of tones (`monitorBand`) and the total event rate are saved in `OUTPUT_FILE` (MATLAB `.mat` format).

Make sure that the audio output of the host is connected to the NAS audio input and that no other application is using the sound card before running the next cell. It takes roughly `NUM_FREQS` x (`SETTLE_TIME` + `MONITOR_TIME` + tone tail) seconds.

In [ ]:
import sounddevice as sd
from scipy.io import savemat

# ==============================
# Experiment parameters
# ==============================
NUM_FREQS = 128              # number of tones in the sweep
NAS_CHANNELS = 64            # channels per cochlea
START_FREQ = 16              # Hz
END_FREQ = 28000             # Hz
FS = 96000                   # audio sampling rate (Hz)
OKT_FREQ = 1e8               # OKAERTool timestamp clock (Hz): 1 tick = 10 ns
GAIN = 0.8                   # tone amplitude (0..1)
MONITOR_TIME = 0.5           # seconds of NAS activity recorded per tone
SETTLE_TIME = 0.2            # seconds between starting the tone and starting the recording
TONE_TIME = 1.0              # duration of the generated tone (must exceed SETTLE_TIME + MONITOR_TIME)
MAX_RETRIES = 3              # maximum attempts per tone when a dropout is detected
DROP_RATIO = 0.8             # a rate below DROP_RATIO * previous rate is considered a dropout
RATE_FLOOR = 10              # spikes/s assigned to silent channels (log-scale friendly)
USB_TRANSFER_LENGTH = 128 * 1024
OUTPUT_FILE = "FrequencySweep.mat"

monitorBand = np.logspace(np.log10(START_FREQ), np.log10(END_FREQ), NUM_FREQS)

okaer.USB_TRANSFER_LENGTH = USB_TRANSFER_LENGTH
sd.default.samplerate = FS
sd.default.channels = 2
sd.default.dtype = 'float32'
sd.default.latency = ('high', 'high')
sd.default.blocksize = 2048


def generate_tone(freq, duration, samplerate=FS, gain=GAIN):
    """Stereo (identical channels) sine wave of the given frequency."""
    t = np.arange(int(duration * samplerate)) / samplerate
    mono = (gain * np.sin(2 * np.pi * freq * t)).astype(np.float32)
    return np.column_stack((mono, mono))


def acquire_events(freq):
    """Play a tone and record the NAS output. Returns (addresses, timestamps) as numpy arrays."""
    tone = generate_tone(freq, TONE_TIME)
    okaer.reset_board(mode='internal')
    sd.play(tone, samplerate=FS)
    try:
        time.sleep(SETTLE_TIME)
        spikes = okaer.monitor(inputs=['port_a'], duration=MONITOR_TIME)
    finally:
        sd.wait()
    if spikes is None:
        return np.empty(0, dtype=np.int64), np.empty(0, dtype=np.int64)
    return np.asarray(spikes[0].addresses, dtype=np.int64), np.asarray(spikes[0].timestamps, dtype=np.float64)


def event_rate(timestamps):
    """Total event rate (spikes/s) of a recording."""
    if len(timestamps) < 2 or timestamps[-1] <= timestamps[0]:
        return 0.0
    return len(timestamps) / ((timestamps[-1] - timestamps[0]) / OKT_FREQ)


def cochlea_responses(addresses, timestamps, n_ch=NAS_CHANNELS):
    """Per-channel firing rates (spikes/s) for the left and right cochleae."""
    rates = np.full(2 * n_ch, float(RATE_FLOOR))
    if len(timestamps) >= 2 and timestamps[-1] > timestamps[0]:
        window = (timestamps[-1] - timestamps[0]) / OKT_FREQ
        channels = addresses // 2                      # drop the polarity bit
        channels = channels[(channels >= 0) & (channels < 2 * n_ch)]
        counts = np.bincount(channels, minlength=2 * n_ch)
        rates = np.where(counts > 0, counts / window, RATE_FLOOR)
    return rates[:n_ch], rates[n_ch:]


# ==============================
# Frequency sweep
# ==============================
tfL, tfR, eventRate = [], [], []
n_dropouts = 0

try:
    for k, freq in enumerate(monitorBand, start=1):
        for attempt in range(1, MAX_RETRIES + 1):
            addresses, timestamps = acquire_events(freq)
            current_rate = event_rate(timestamps)
            is_dropout = current_rate == 0 or (len(eventRate) > 0 and current_rate < DROP_RATIO * eventRate[-1])
            if not is_dropout:
                break
            n_dropouts += 1
            print(f"  Dropout detected at {freq:.2f} Hz (attempt {attempt}/{MAX_RETRIES})")
        print(f"[{k:3d}/{NUM_FREQS}] {freq:9.2f} Hz: {len(timestamps)} spikes, {current_rate:.0f} spikes/s")

        left, right = cochlea_responses(addresses, timestamps)
        tfL.append(left)
        tfR.append(right)
        eventRate.append(current_rate)
finally:
    sd.stop()

# ==============================
# Save results
# ==============================
savemat(OUTPUT_FILE, {
    "numberOfBands": NUM_FREQS,
    "startFreq": START_FREQ,
    "endFreq": END_FREQ,
    "monitorBand": monitorBand[:len(tfL)],
    "tfL": np.array(tfL),
    "tfR": np.array(tfR),
    "eventRate": np.array(eventRate),
})
print(f"\nSweep finished ({n_dropouts} dropouts detected). Results saved in {OUTPUT_FILE}")

### Analysis of the characterization results
The cell below loads the `.mat` file produced by the sweep and generates the following figures:

1. **Frequency activity** (left and right): firing rate of every NAS channel versus tone frequency, i.e. the tuning curve of each channel. Hovering the mouse over a curve shows the channel number when `mplcursors` is installed.
2. **Frequency response maps** (left and right): the same data as a colour map (NAS channel vs. tone frequency), where the diagonal band shows how channel centre frequencies decrease along the cascade.
3. **Centre frequency error**: the centre frequency of each channel is estimated as the tone that maximizes its firing rate and compared with the nominal logarithmic distribution between 22 kHz and 20 Hz. The mean relative error is printed and the measured/nominal curves are plotted.
4. **Peak activity trend**: the maximum firing rate of channels 5-25 is fitted with a logarithmic model (linear regression on `log(channel)`) and extrapolated to channels 26-64, showing how the peak activity evolves along the cascade.

In [ ]:
import matplotlib.pyplot as plt
from scipy.io import loadmat
from sklearn.linear_model import LinearRegression

try:
    import mplcursors
except ImportError:
    mplcursors = None

# ==============================
# Load data
# ==============================
data = loadmat(OUTPUT_FILE)
tfL = data["tfL"]
tfR = data["tfR"]
monitorBand = data["monitorBand"].squeeze()
startFreq = float(data["startFreq"].squeeze())
endFreq = float(data["endFreq"].squeeze())
NASBands = tfL.shape[1]
channels = np.arange(1, NASBands + 1)
print(f"Loaded {tfL.shape[0]} tones x {NASBands} channels per cochlea from {OUTPUT_FILE}")

sides = (("Left", tfL), ("Right", tfR))

# ==============================
# 1. Frequency activity (tuning curves)
# ==============================
fig, axs = plt.subplots(2, 1, figsize=(10, 12), constrained_layout=True)
for ax, (name, tf) in zip(axs, sides):
    lines = ax.semilogx(monitorBand, tf, lw=1)
    for ch, line in zip(channels, lines):
        line.set_label(f"Channel {ch}")
    ax.set(title=f"Frequency Activity ({name})", xlabel="Audio frequency (Hz)",
           ylabel=f"Event Rate (evt/sec) {name}", xlim=(20, 20000))
    ax.grid(True)
    if mplcursors is not None:
        mplcursors.cursor(lines, hover=True).connect("add", lambda sel: sel.annotation.set_text(sel.artist.get_label()))
plt.show()

# ==============================
# 2. Frequency response maps
# ==============================
fig, axs = plt.subplots(2, 1, figsize=(10, 12), constrained_layout=True)
for ax, (name, tf) in zip(axs, sides):
    pcm = ax.pcolormesh(channels, monitorBand, tf, shading="auto", cmap="viridis")
    ax.set_yscale("log")
    ax.set(title=f"Frequency Response ({name})", xlabel="NAS Channel",
           ylabel="Audio frequency (Hz)", ylim=(startFreq, endFreq))
    fig.colorbar(pcm, ax=ax, label="Spikes Activity (spikes/sec)")
plt.show()

# ==============================
# 3. Centre frequency error (nominal: log-spaced from 22 kHz down to 20 Hz)
# ==============================
targetFreq = np.logspace(np.log10(22e3), np.log10(20), NASBands)
fig, ax = plt.subplots(figsize=(10, 5))
for name, tf in sides:
    wc = monitorBand[np.argmax(tf, axis=0)]
    errorFreq = np.abs(targetFreq - wc) / targetFreq
    print(f"{name}: mean relative centre-frequency error = {100 * errorFreq.mean():.2f} %")
    ax.semilogy(channels, wc, 'o-', ms=3, label=f"Measured ({name})")
ax.semilogy(channels, targetFreq, 'k--', label="Nominal")
ax.set(title="Channel centre frequency", xlabel="NAS Channel", ylabel="Frequency (Hz)")
ax.grid(True, which="both")
ax.legend()
plt.show()

# ==============================
# 4. Peak activity trend: logarithmic fit of channels 5-25, extrapolated to 26-64
# ==============================
channels_train = np.arange(5, 26)
channels_predict = np.arange(26, NASBands + 1)
fig, axes = plt.subplots(2, 1, figsize=(12, 10), sharex=True)
for ax, (name, tf), color in zip(axes, sides, ("b", "g")):
    peak_train = tf[:, channels_train - 1].max(axis=0)
    model = LinearRegression().fit(np.log(channels_train).reshape(-1, 1), peak_train)
    predicted = model.predict(np.log(channels_predict).reshape(-1, 1))
    for ch in channels:
        ax.plot(np.full(tf.shape[0], ch), tf[:, ch - 1], 'k.', alpha=0.3)
    ax.plot(channels_train, peak_train, f'{color}o-', label="Peak activity, channels 5-25")
    ax.plot(channels_predict, predicted, 'r--', label=f"Prediction, channels 26-{NASBands}")
    ax.set(title=f"Peak activity trend and prediction ({name}, logarithmic model)",
           ylabel="Peak activity (spikes/sec)")
    ax.grid(True)
    ax.legend()
axes[-1].set_xlabel("NAS Channel")
plt.show()

### Bypassing the NAS output to the OKAERTool output data bus
Once the characterization is done, the NAS output can be bypassed to the OKAERTool output data bus using the `bypass` method of the OKAERTool class. This is useful to forward the NAS events to another device without any OKAERTool processing.

In [ ]:
okaer.bypass(inputs=['port_a'])